## 1. Scratchpad Final

In [51]:
import pickle, copy, gc
from tqdm.notebook import tqdm
import BPE_tokenizer
from BPE_tokenizer import BPETokenizer
from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
with open("../data/tokenizer_euro_pol.pkl", 'rb') as f:
    tokenizer_pol = pickle.load(f)

with open("../data/tokenizer_euro_eng.pkl", 'rb') as f:
    tokenizer_eng = pickle.load(f)

with open("../data/dataframe_europarl.pkl", 'rb') as f:
    df_final = pickle.load(f)

In [76]:
class BPEEncoder():
    def __init__(self, bpe_vocab, bpe_vocab_chrs, thres_tup=55, is_tgt=False):
        self.vocab_encoder = {"".join(k): v for k, v in bpe_vocab.items()}
        self.word_encoder = {"".join(k): [self.vocab_encoder.get(x) for x in k] for k, _ in bpe_vocab_chrs}
        self.vocab_tuple = list(bpe_vocab.keys())[thres_tup:]
        self.char_factor = lambda word: [x for x in word] + ['_']

    def encode_snt(self, snt_toks):
        return [y for x in snt_toks for y in self.encode_word(x)]

    def encode_word(self, word):
        word_factor = self.char_factor(word)
        word_id = self.word_encoder.get("".join(word_factor), None)
        if word_id is not None:
            yield from word_id
        else:
            word_pairs = zip(word_factor, word_factor[1:])
            for pair in self.vocab_tuple:
                if pair in word_pairs:
                    p_1, p_2 = map(re.escape, pair)
                    word_factor = re.sub(rf"(?<=\s){p_1}\s{p_2}(?=\s)", f"{pair[0]}{pair[1]}", f" {' '.join(word_factor)} ").split()
                    word_pairs = zip(word_factor, word_factor[1:])
            yield from [self.vocab_encoder[x] for x in word_factor]

In [77]:
encoder_eng = BPEEncoder(tokenizer_eng.vocab, tokenizer_eng.vocab_chrs)
encoder_pol = BPEEncoder(tokenizer_pol.vocab, tokenizer_pol.vocab_chrs, 65, True)

In [82]:
encoder_pol2 = BPE_tokenizer.BPEEncoder(tokenizer_pol.vocab, 65, True)

In [59]:
df_eng = copy.deepcopy(df_final[['eng_text', 'eng_ids']])
df_pol = copy.deepcopy(df_final[['pol_text', 'pol_ids']])

In [88]:
samp = df_pol.sample(1000)
all(samp['pol_text'].progress_apply(encoder_pol2.encode_snt)==samp['pol_split'].progress_apply(encoder_pol.encode_snt))

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

True

In [78]:
df_eng['eng_split'].progress_apply(lambda x: encoder_eng.encode_snt(x) + [2]) == df_eng['eng_ids']
df_pol['pol_split'].progress_apply(lambda x: [4]+encoder_pol.encode_snt(x) + [2]) == df_pol['pol_ids']

  0%|          | 0/576824 [00:00<?, ?it/s]

  0%|          | 0/576824 [00:00<?, ?it/s]

0         True
1         True
2         True
3         True
4         True
          ... 
576819    True
576820    True
576821    True
576822    True
576823    True
Length: 576824, dtype: bool

In [50]:
all(df_eng['eng_split'].apply(lambda x: encoder_eng1.encode_snt(x) + [2]) == df_eng['eng_ids'])

True

In [58]:
df_final

,eng_text,eng_ids,pol_text,pol_ids
0,Action taken on Parliament's resolutions: see ...,"[771, 909, 73, 326, 258, 3994, 434, 790, 1791, 2]",Działania podjęte w wyniku rezolucji Parlament...,"[4, 756, 4667, 78, 2561, 1493, 764, 520, 2726,..."
1,Documents received: see Minutes,"[2769, 2649, 434, 790, 1791, 2]",Składanie dokumentów: patrz protokół,"[4, 9904, 3928, 520, 2726, 2578, 2]"
2,Texts of agreements forwarded by the Council: ...,"[4207, 78, 1458, 9696, 201, 61, 429, 434, 790,...",Teksty porozumień przekazane przez Radę: patrz...,"[4, 11009, 5057, 10035, 280, 2657, 520, 2726, ..."
3,Membership of Parliament: see Minutes,"[3303, 78, 326, 434, 790, 1791, 2]",Skład Parlamentu: patrz protokół,"[4, 6156, 764, 520, 2726, 2578, 2]"
4,Membership of committees and delegations: see ...,"[3303, 78, 3853, 81, 5881, 434, 790, 1791, 2]",Skład komisji i delegacji: patrz protokół,"[4, 6156, 488, 67, 5201, 520, 2726, 2578, 2]"
...,...,...,...,...
576819,Composition of committees and delegations : se...,"[5621, 78, 3853, 81, 5881, 434, 790, 1791, 2]",Skład komisji i delegacji: Patrz protokól,"[4, 6156, 488, 67, 5201, 520, 2726, 12347, 2]"
576820,Agenda of the next sitting : see Minutes,"[1893, 78, 61, 1130, 2342, 434, 790, 1791, 2]",Porządek obrad następnego posiedzenia: Patrz p...,"[4, 6712, 4028, 7678, 2618, 520, 2726, 12347, 2]"
576821,Closure of the sitting,"[4870, 78, 61, 2342, 2]",Zamknięcie posiedzenia,"[4, 7782, 2618, 2]"
576822,(The sitting closed at 22.25),"[891, 61, 2342, 2295, 90, 4925, 74, 2972, 1014...",(The sitting closed at 22.25),"[4, 1064, 14183, 115, 11961, 5637, 57, 5366, 5..."


In [43]:
df_eng.head()

,eng_text,eng_ids,eng_split
319152,There is a great deal of irritation that the c...,"[294, 83, 91, 1144, 1529, 78, 242, 693, 136, 1...","[there, is, a, great, deal, of, irritation, th..."
406221,I have tried to answer most of the questions t...,"[125, 167, 4371, 80, 2326, 584, 78, 61, 1849, ...","[i, have, tried, to, answer, most, of, the, qu..."
111117,To say nothing of the fact that accession nego...,"[80, 738, 2076, 78, 61, 700, 103, 2153, 1302, ...","[to, say, nothing, of, the, fact, that, access..."
561843,"Equally, the level of involvement particularly...","[3581, 68, 61, 721, 78, 3125, 935, 78, 1303, 8...","[equally, ,, the, level, of, involvement, part..."
193161,The employment rates of older people have rise...,"[61, 1108, 3277, 78, 5351, 416, 167, 7797, 85,...","[the, employment, rates, of, older, people, ha..."
